In [1]:
# Create a `QueryEngine` for retrieval augmented generation
#
# Setting up the persona database
#
# We will be using personas from the 
# [dvilasuero/finepersonas-v0.1-tiny dataset](https://huggingface.co/datasets/dvilasuero/finepersonas-v0.1-tiny).
# This dataset contains 5K personas that will be attending the party!
#
# Let's load the dataset and store it as files in the `data` directory
#
# NOTE: Only run once to bootstrap TXT files onto disk
#

from datasets import load_dataset
from pathlib import Path

dataset = load_dataset(path="dvilasuero/finepersonas-v0.1-tiny", split="train")

Path("data").mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(Path("data") / f"persona_{i}.txt", "w") as f:
        f.write(persona["persona"])

In [2]:
# Awesome, now we have a local directory with all the personas that will be attending the party, we can load and index!
#
# Loading and embedding persona documents
#
# We will use the `SimpleDirectoryReader` to load the persona descriptions from the `data` directory. 
# This will return a list of `Document` objects. 
#

from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()
len(documents)

5000

In [3]:
# using Ollama embedding so set up here once

from llama_index.embeddings.ollama import OllamaEmbedding

BGE_SMALL_EN_V15_Q4_K_M = "qllama/bge-small-en-v1.5:q4_k_m"

MODEL = BGE_SMALL_EN_V15_Q4_K_M

ollama_embedding = OllamaEmbedding(
    model_name=MODEL,
    base_url="http://localhost:11434",
    ollama_additional_kwargs={"mirostat": 0},
)

In [4]:
# Now we have a list of `Document` objects, we can use 
# the `IngestionPipeline` to create nodes from the documents and prepare them for the `QueryEngine`. 
#
# We will use the `SentenceSplitter` to split the documents into smaller chunks and the `HuggingFaceEmbedding` to embed the chunks.
#

if not ollama_embedding:
    raise ValueError("ollama_embedding object not set")

if not documents:
    raise ValueError('documents object not set')

#from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline


# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        #HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
        ollama_embedding,
    ]
)

# run the pipeline sync or async
nodes = await pipeline.arun(documents=documents[:10])
nodes

[TextNode(id_='7208d135-5711-4d1e-b558-b536436ed517', embedding=[-0.6243568658828735, -0.19033612310886383, 0.13992750644683838, 0.24617329239845276, 0.2047269195318222, -0.0816047191619873, -0.02997751533985138, 0.24447233974933624, -0.6413572430610657, -0.5155730247497559, -0.07594816386699677, -0.1781136393547058, -0.21331584453582764, 0.525693416595459, -0.2010507583618164, 0.3751751184463501, 0.051098987460136414, 0.6274140477180481, -0.3637396991252899, 0.15681791305541992, -0.011683672666549683, -0.5794965624809265, 0.4565783143043518, -0.27691662311553955, -0.03444599360227585, 0.030798226594924927, 0.07409168779850006, -0.09410659968852997, -0.13280002772808075, -1.0801517963409424, -0.29543337225914, -0.003003247082233429, 0.008680276572704315, 0.038163937628269196, 0.4005179703235626, 0.4089178442955017, 0.19079247117042542, 0.4213290810585022, 0.08515879511833191, 0.37228697538375854, 0.45771580934524536, 0.12524861097335815, -0.4845031201839447, 0.46503448486328125, 0.3513

In [5]:
# As, you can see, we have created a list of `Node` objects, which are just chunks of text from the original documents.
# Let's explore how we can add these nodes to a vector store.

# Storing and indexing documents

# Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it.
# In this case, we will use `Chroma` to store our documents.
# Let's run the pipeline again with the vector store attached. 
# The `IngestionPipeline` caches the operations so this should be fast!
#

if not ollama_embedding:
    raise ValueError("ollama_embedding object not set")

import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        # HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
        ollama_embedding,
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

10

In [6]:
# We can create a `VectorStoreIndex` from the vector store and use it to query
# the documents by passing the vector store and embedding model to the `from_vector_store()` method.

if not ollama_embedding:
    raise ValueError("ollama_embedding object not set")

if not vector_store:
    raise ValueError('vector_store object not set')

from llama_index.core import VectorStoreIndex
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    # embed_model=embed_model
    embed_model=ollama_embedding
)

In [9]:
# We don't need to worry about persisting the index to disk,
# as it is automatically saved within the `ChromaVectorStore` object
# and the passed directory path.
#
### Querying the index
#
# Now that we have our index, we can use it to query the documents.
# Let's create a `QueryEngine` from the index and use it to query the documents using a specific response mode.
#

#from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

if not index:
    raise ValueError('index object not set')

from llama_index.llms.ollama import Ollama
import nest_asyncio

nest_asyncio.apply()  # This is needed to run the query engine

# llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

QWEN25_CODER_14B_Q4_K_M = "qwen2.5-coder:14b"

MODEL = QWEN25_CODER_14B_Q4_K_M

llm = llm = Ollama(
    base_url="http://localhost:11434",
    model=MODEL,
    timeout=0
)

query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
response = query_engine.query(
    "Respond using a persona that describes author and travel experiences?"
)
response

Response(response="I am an anthropologist with a profound interest in Cypriot culture, history, and society. My academic journey has been enriched by years of living and researching on the island, allowing me to immerse myself deeply into its people's customs and lifestyle. This extensive experience has endowed me with a unique perspective that bridges cultural understanding and academic rigor.", source_nodes=[NodeWithScore(node=TextNode(id_='fb303d34-af8f-44ac-b6c7-856442a42b88', embedding=None, metadata={'file_path': '/home/mikejay/Documents/github.com.d/brown-coat/agents-course/notebooks/unit2/llama-index/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2025-06-22', 'last_modified_date': '2025-06-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last

In [11]:
## Evaluation and observability
#
# LlamaIndex provides **built-in evaluation tools to assess response quality.**
# These evaluators leverage LLMs to analyze responses across different dimensions.
# We can now check if the query is faithful to the original persona.
#

if not llm:
    raise ValueError('llm object not set')

if not response:
    raise ValueError('response object not set')

from llama_index.core.evaluation import FaithfulnessEvaluator

# supress timeout
import nest_asyncio
nest_asyncio.apply()

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=response, timeout=0)
eval_result.passing

True

In [12]:
# If one of these LLM based evaluators does not give enough context,
# we can check the response using the Arize Phoenix tool,
# after creating an account at [LlamaTrace](https://llamatrace.com/login) and generating an API key.
#
# NOTE: using local Phoenix via docker compose
#

import llama_index
import os

# PHOENIX_API_KEY = "<PHOENIX_API_KEY>"
# os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"

# supress timeout
import nest_asyncio
nest_asyncio.apply()

llama_index.core.set_global_handler(
    "arize_phoenix",
    # endpoint="https://llamatrace.com/v1/traces"
    endpoint="http://localhost:6006/v1/traces",    
)


In [15]:
# Now, we can query the index and see the response in the Arize Phoenix tool.

if not query_engine:
    raise ValueError('query_engine object not set')

# supress timeout
import nest_asyncio
nest_asyncio.apply()

response = query_engine.query(
    "What is the name of the someone that is interested in AI and techhnology?"
)
response

Response(response='The query asks for the name of someone interested in AI and technology, but the provided context does not mention any specific person with such interests. The information given pertains to a pulmonologist or respiratory specialist who focuses on educating patients about the respiratory system and its diseases. There is no connection made between this medical professional and an interest in AI or technology.', source_nodes=[NodeWithScore(node=TextNode(id_='97634a3f-a31f-4ddd-a3cd-f4fd60cc7bd2', embedding=None, metadata={'file_path': '/home/mikejay/Documents/github.com.d/brown-coat/agents-course/notebooks/unit2/llama-index/data/persona_1000.txt', 'file_name': 'persona_1000.txt', 'file_type': 'text/plain', 'file_size': 133, 'creation_date': '2025-06-22', 'last_modified_date': '2025-06-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type

We can then go to the [LlamaTrace](https://llamatrace.com/login) and explore the process and response.

![arize-phoenix](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/arize.png)    